# 🛢️ Nodal Analysis & Wellbore Simulation in Google Colab

Welcome to the interactive Nodal Analysis simulation notebook! This notebook allows you to model petroleum well performance by combining **Inflow Performance Relationship (IPR)** and **Vertical Lift Performance (VLP)** curves.

### 📌 How to use in Google Colab:
1. **Upload Files (Option 2):** Upload this `.ipynb` file along with the `nodal/` folder (containing `fluid_properties.py`, `ipr.py`, `vlp.py`, etc.) into the Colab file explorer on the left. *(Don't worry—if you forgot to upload `nodal/`, Cell 1 will automatically fetch it from GitHub as a fallback!)*
2. **Run Cells:** Click the play button on each cell sequentially from top to bottom.

In [ ]:
# =====================================================================
# STEP 1: Install Requirements & Check Environment
# =====================================================================
!pip install -q numpy scipy plotly pandas

import os
import sys

# Check if 'nodal' package exists locally (Option 2: uploaded manually)
if not os.path.exists("nodal"):
    print("📥 'nodal' package not found locally. Fetching automatically from GitHub as fallback...")
    !git clone -q https://github.com/lolamlol1/Nodal_analysis_project.git temp_repo
    !mv temp_repo/nodal ./nodal
    !rm -rf temp_repo
    print("✅ Successfully downloaded 'nodal' package!")
else:
    print("✅ Found local 'nodal' package from uploaded files!")

# Add current directory to Python path
if '.' not in sys.path:
    sys.path.append('.')

print("🚀 Environment ready! Proceed to the next cell.")

## 1️⃣ Define Reservoir & Fluid Properties (PVT)
We initialize the `FluidProperties` class with field data such as API gravity, Gas Specific Gravity, Gas-Oil Ratio (GOR), Water Cut, and Reservoir Temperature.

In [ ]:
from nodal.fluid_properties import FluidProperties

# Initialize PVT properties
fp = FluidProperties(
    api_gravity=35.0,     # °API
    gas_sg=0.65,          # Air = 1.0
    water_sg=1.07,        # Fresh water = 1.0
    gor=500.0,            # Producing GOR (scf/STB)
    water_cut=0.20,       # 20% water cut
    reservoir_temp=180.0  # °F
)

print(f"🛢️ Oil Specific Gravity: {fp.oil_sg:.4f}")
print(f"🫧 Bubble-Point Pressure: {fp.bubble_point:.1f} psia")
print(f"💧 Oil FVF at 3000 psia:  {fp.oil_fvf(3000.0, fp.T_res):.4f} RB/STB")
print(f"🍯 Oil Viscosity at 3000 psia: {fp.oil_viscosity(3000.0, fp.T_res):.2f} cp")

## 2️⃣ Initialize IPR and VLP Calculators
Next, we set up the inflow performance (reservoir to wellbore) and outflow performance (wellbore to surface).
- **IPR Model:** Vogel (supported models: `vogel`, `jones`, `fetkovitch`, `standing`)
- **VLP Correlation:** Beggs & Brill (supported: `beggs_brill`, `hagedorn_brown`)

In [ ]:
from nodal.ipr import IPRCalculator
from nodal.vlp import VLPCalculator

# Well & Reservoir Parameters
reservoir_pressure = 4200.0  # psia
productivity_index = 2.5     # STB/day/psi
tvd = 8000.0                 # True Vertical Depth (ft)
md = 9000.0                  # Measured Depth (ft)
tubing_id = 2.992            # Tubing Inner Diameter (inches - e.g. 3-1/2" tubing)
whp = 250.0                  # Wellhead Pressure (psia)

# Initialize calculators
ipr_calc = IPRCalculator(fp, reservoir_press=reservoir_pressure, pi=productivity_index, model="vogel")
vlp_calc = VLPCalculator(fp, tvd=tvd, md=md, tubing_id=tubing_id, whp=whp)

print(f"📉 Absolute Open Flow (AOF): {ipr_calc.aof('vogel'):.0f} STB/day")

## 3️⃣ Solve for Operating Point & Plot IPR/VLP Curves
We combine the IPR and VLP curves using `NodalAnalysis` to find the exact equilibrium operating point (Flow Rate $q$ and Flowing Bottom-Hole Pressure $P_{wf}$). We then visualize the curves using an interactive Plotly chart!

In [ ]:
from nodal.nodal_solver import NodalAnalysis
import plotly.graph_objects as go

# Solve Nodal Analysis
solver = NodalAnalysis(ipr_calc, vlp_calc)
op = solver.operating_point(ipr_model="vogel", vlp_correlation="beggs_brill")

print("="*60)
print(f"🎯 OPERATING POINT RESULTS:")
print(f"   • Flow Rate (q): {op['q']:.1f} STB/day")
print(f"   • Flowing Bottom-Hole Pressure (Pwf): {op['Pwf']:.1f} psia")
print(f"   • Status: {op['message']}")
print("="*60)

# Generate curve arrays for plotting
q_ipr, pwf_ipr = ipr_calc.ipr_curve("vogel")
q_vlp, pwf_vlp = vlp_calc.vlp_curve(q_max=op["aof"]*1.1, correlation="beggs_brill")

# Create interactive Plotly figure
fig = go.Figure()

# Add IPR Curve
fig.add_trace(go.Scatter(
    x=q_ipr, y=pwf_ipr, mode='lines', name='IPR (Inflow - Vogel)',
    line=dict(color='#00d2ff', width=3)
))

# Add VLP Curve
fig.add_trace(go.Scatter(
    x=q_vlp, y=pwf_vlp, mode='lines', name='VLP (Outflow - Beggs & Brill)',
    line=dict(color='#ff4b4b', width=3)
))

# Add Operating Point Marker
if op["found"]:
    fig.add_trace(go.Scatter(
        x=[op["q"]], y=[op["Pwf"]], mode='markers+text', name='Operating Point',
        marker=dict(color='#00ff88', size=14, line=dict(color='white', width=2)),
        text=[f"  q={op['q']:.0f}<br>  Pwf={op['Pwf']:.0f}"],
        textposition="top right"
    ))

fig.update_layout(
    title="<b>Nodal Analysis: IPR vs VLP Curve Intersection</b>",
    xaxis_title="<b>Flow Rate (STB/day)</b>",
    yaxis_title="<b>Flowing Bottom-Hole Pressure (psia)</b>",
    template="plotly_dark",
    hovermode="x unified",
    legend=dict(x=0.65, y=0.95, bgcolor="rgba(0,0,0,0.5)"),
    width=900, height=550
)

fig.show()

## 4️⃣ Sensitivity Study: Comparing Tubing Sizes
In well design, selecting the right tubing size is critical. A smaller tubing creates too much friction at high rates, while a larger tubing may lead to liquid loading or slippage. Here we compare four standard tubing sizes against our IPR curve!

In [ ]:
# Compare standard tubing sizes
tubing_sizes = {
    '2-3/8" (1.995" ID)': 1.995,
    '2-7/8" (2.441" ID)': 2.441,
    '3-1/2" (2.992" ID)': 2.992,
    '4-1/2" (3.958" ID)': 3.958
}

multi_vlp_data = vlp_calc.multi_tubing_vlp(
    tubing_sizes=tubing_sizes,
    q_max=op["aof"]*1.1,
    correlation="beggs_brill"
)

# Plot Tubing Sensitivity
fig_sens = go.Figure()

# Add IPR Curve
fig_sens.add_trace(go.Scatter(
    x=q_ipr, y=pwf_ipr, mode='lines', name='IPR (Vogel)',
    line=dict(color='#00d2ff', width=4, dash='dash')
))

# Add VLP curves for each tubing size
colors = ['#ffaa00', '#ff4b4b', '#b54bff', '#00ff88']
for idx, (label, data) in enumerate(multi_vlp_data.items()):
    fig_sens.add_trace(go.Scatter(
        x=data["q"], y=data["fbhp"], mode='lines', name=f'VLP {label}',
        line=dict(color=colors[idx % len(colors)], width=2.5)
    ))

fig_sens.update_layout(
    title="<b>Tubing Size Sensitivity Analysis</b>",
    xaxis_title="<b>Flow Rate (STB/day)</b>",
    yaxis_title="<b>Flowing Bottom-Hole Pressure (psia)</b>",
    template="plotly_dark",
    hovermode="x unified",
    width=900, height=550
)

fig_sens.show()